# Modelo HAR: Entrenamiento y Evaluación

In [1]:
import os
import random
import sys
import warnings
from pathlib import Path
import optuna
import mlflow
import time

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))

from src.models.har_wrapper import HARWrapper
from src.eval.scorer import VolatilityScorer
from src.trainer.walk_forward import WalkForwardTrainer
from src.utils import trainer_toolkit as tt
from src.utils import ts_toolkit as tst
from src.utils import test_toolkit as ts

warnings.filterwarnings("ignore")

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)

mlflow.set_tracking_uri("sqlite:///mlflow.db")

OUT_PATH = PROJECT_ROOT / "data" / "processed"
RESULTS_PATH = PROJECT_ROOT / "results" / "har"
MODELS_PATH = PROJECT_ROOT / "models"

RESULTS_PATH.mkdir(parents=True, exist_ok=True)
MODELS_PATH.mkdir(parents=True, exist_ok=True)

df_train = pd.read_parquet(OUT_PATH / "brent_train.parquet")
y_train = df_train["log_returns"].values.astype("float64")
rs_train = df_train["rs_daily"].values.astype("float64")

df_test = pd.read_parquet(OUT_PATH / "brent_test.parquet")
y_test = df_test["log_returns"].values.astype("float64")
rs_test = df_test["rs_daily"].values.astype("float64")

assert not np.isnan(y_train).any() and not np.isnan(rs_train).any(), "NaNs en train"
assert not np.isnan(y_test).any() and not np.isnan(rs_test).any(), "NaNs en test"
assert df_train.index.max() < df_test.index.min(), "Solapamiento temporal train/test"

print("=" * 70)
print(f"Train: {len(y_train)} obs  ({df_train.index[0].date()} -> {df_train.index[-1].date()})")
print(f"Test:  {len(y_test)} obs  ({df_test.index[0].date()} -> {df_test.index[-1].date()})")
print("=" * 70)

Train: 2171 obs  (2014-01-03 -> 2022-11-03)
Test:  953 obs  (2022-11-07 -> 2026-08-21)


:::{admonition} Resumen
:class: info
- La validación *Walk-Forward* seleccionó el HAR ($L_w = 5$, $L_m = 22$, QLIKE $0.6066$) estimado por QR reducida con errores robustos HAC.
- La ventana de 22 días superó a la de 30 días, en línea con Corsi (2009), con persistencia elevada y reversión a la media (suma $0.8490$).
- Fuera de muestra mantuvo estabilidad en QLIKE ($0.4471$), sin autocorrelación ni efectos ARCH tras comparaciones múltiples y con ajuste a una *t*-Student ($\nu = 6.0896$).
- Persisten desviaciones de normalidad y dependencia no lineal en el contraste BDS, lo que motiva pasar a arquitecturas híbridas como WTD-HAR-GRU.
- Mantiene el suavizado ante saltos bruscos y no alcanza el pico de marzo de 2026, asociado como hipótesis a las ventanas de promedio.
:::

## Introducción

El modelo GARCH y sus extensiones (como el EGARCH) capturan la persistencia de la volatilidad a través de los rezagos de la varianza condicional y de los choques recientes. Sin embargo, la literatura empírica ha demostrado que la volatilidad de los activos financieros exhibe una estructura de memoria larga que los modelos GARCH tradicionales no logran capturar eficientemente sin imponer restricciones de estacionariedad que rozan la integración fraccional. 

Para abordar esta limitación, Corsi (2009) propone el modelo Heterogeneous Autoregressive (HAR). Basado en la Hipótesis de Mercado Heterogéneo (Haugen, 1995; Müller et al., 1997), el HAR asume que el mercado está compuesto por agentes que operan a distintos horizontes temporales (intradía, semanal y mensual), lo que genera una estructura de volatilidad con múltiples frecuencias. Esta sección presenta, estima y evalúa el modelo HAR bajo el mismo protocolo experimental empleado para los modelos GARCH, con el fin de garantizar la comparabilidad de sus resultados.

### Modelo HAR Original (HAR-RV)

En su formulación original, el HAR modela la volatilidad realizada (RV) como una función lineal de sus valores pasados agregados en tres escalas temporales:

$$
RV_t = c + \beta_d RV_{t-1}^{(d)} + \beta_w RV_{t-1}^{(w)} + \beta_m RV_{t-1}^{(m)} + \varepsilon_t
$$

Donde:
- $RV_{t-1}^{(d)} = RV_{t-1}$ representa la volatilidad realizada diaria.
- $RV_{t-1}^{(w)} = \frac{1}{5}\sum_{i=1}^{5} RV_{t-i}$ representa el promedio de la volatilidad de la última semana (5 días hábiles).
- $RV_{t-1}^{(m)} = \frac{1}{22}\sum_{i=1}^{22} RV_{t-i}$ representa el promedio de la volatilidad del último mes (22 días hábiles).

Los coeficientes $\beta_d$, $\beta_w$ y $\beta_m$ capturan la influencia de los agentes de corto, mediano y largo plazo, respectivamente. Este modelo aproxima la memoria larga mediante una estructura autorregresiva simple, requiriendo únicamente la estimación de tres parámetros.

### Adaptación Metodológica: HAR sobre Retornos (HAR-RS)

Para mantener la consistencia metodológica estricta exigida por el benchmark de este trabajo, el modelo HAR original debe ser adaptado. Los modelos paramétricos (GARCH, EGARCH) y los modelos de Deep Learning (GRU, LSTM, TCN) empleados en este estudio tienen dos características en común: (i) reciben exclusivamente la serie de retornos históricos sin transformar como *input*, elevando internamente los retornos al cuadrado para formar los regresores, y (ii) su variable objetivo (*target*) es la varianza de Rogers-Satchell (RS).

Por lo tanto, se descarta el uso de la volatilidad realizada intradía ($RV$) como insumo externo, y se adapta el HAR para operar bajo las mismas condiciones. El modelo, denotado como HAR-RS, se especifica de la siguiente manera:

$$
\widehat{RS}_t = c + \beta_d r_{t-1}^2 + \beta_w \left(\frac{1}{L_w}\sum_{i=1}^{L_w} r_{t-i}^2\right) + \beta_m \left(\frac{1}{L_m}\sum_{i=1}^{L_m} r_{t-i}^2\right) + \varepsilon_t
$$

Donde $r_{t-i}$ representa el retorno logarítmico diario. Al elevar los retornos al cuadrado ($r_{t-i}^2$), el modelo utiliza el cuadrado de los retornos como un proxy de la volatilidad pasada. Bajo el supuesto de que la media condicional de los retornos es cero ($\mu_t = 0$), se cumple que $\mathbb{E}_{t-1}[r_t^2] = \sigma_t^2$, lo que valida teóricamente el uso del retorno al cuadrado como estimador insesgado de la varianza condicional latente. La variable dependiente ($\widehat{RS}_t$) es la varianza de Rogers-Satchell, garantizando que el modelo predice exactamente el mismo *target* que los modelos de Deep Learning.

Si bien la formulación clásica de Corsi (2009) define $L_w=5$ y $L_m=22$, en esta implementación los órdenes de los rezagos semanal ($L_w$) y mensual ($L_m$) se tratan como hiperparámetros sujetos a optimización bayesiana (Optuna), permitiendo al modelo explorar ventanas como 7 y 30 días. Esto permite adaptar la estructura de memoria a las particularidades del mercado del petróleo Brent. Asimismo, para prevenir estrictamente cualquier fuga de información (*data leakage*), todas las agregaciones se calculan aplicando un rezago de un periodo (`shift(1)`) sobre los retornos al cuadrado, garantizando que la predicción en $t$ solo utilice información disponible hasta $t-1$.

### Estimación y Supuestos

La estimación puntual de los parámetros del modelo HAR-RS se obtiene resolviendo el problema de mínimos cuadrados lineales mediante descomposición QR reducida, con respaldo en mínimos cuadrados (`lstsq`) ante mal condicionamiento numérico. Este enfoque sigue siendo válido y es el estándar en la literatura (Corsi, 2009) bajo los siguientes supuestos fundamentales:

1. **Linealidad y Correcta Especificación:** La relación entre la varianza RS y los proxies de volatilidad pasada es lineal en los parámetros.
2. **Media Condicional del Error Cero:** El término de error $\varepsilon_t$ satisface $\mathbb{E}[\varepsilon_t \mid \mathcal{F}_{t-1}] = 0$, lo que implica que el modelo no presenta sesgo sistemático.
3. **No Multicolinealidad Perfecta:** Aunque las medias móviles diaria, semanal y mensual están altamente correlacionadas, no son perfectamente colineales, permitiendo resolver el sistema de forma estable mediante descomposición QR.
4. **Estacionariedad:** La serie de retornos al cuadrado debe ser estacionaria para que los coeficientes estimados sean consistentes y las predicciones no diverjan. Aunque los retornos al cuadrado suelen exhibir autocorrelación serial (induciendo heterocedasticidad condicional en los residuos), la estimación lineal sigue proporcionando estimadores insesgados y consistentes para el pronóstico.

Dado que las series financieras suelen presentar heterocedasticidad condicional y autocorrelación en los residuos, la inferencia estadística sobre los coeficientes se realiza reportando errores estándar consistentes con heterocedasticidad y autocorrelación (HAC), siguiendo la corrección de Newey-West con un máximo de rezagos por defecto determinado por $\max\left(1, \lfloor 4(N/100)^{2/9} \rfloor\right)$ y acotado a $N-1$ (o el valor configurado en `hac_lags`), además de intervalos de confianza por *block bootstrap* cuando se solicitan. Si bien esta corrección no altera los coeficientes puntuales utilizados para el pronóstico, garantiza que los p-valores y los intervalos de confianza reportados en la tabla de resultados sean estadísticamente robustos.

El modelo se implementa computacionalmente en `HARWrapper` (`HARQRFitResult`) utilizando operaciones matriciales de `NumPy` y `SciPy` para la descomposición QR reducida y la inferencia HAC/Newey-West, en lugar de `statsmodels` `OLS`. El pronóstico se genera de forma recursiva (ventana expansiva) bajo el protocolo de validación *walk-forward*, reestimando los parámetros en cada paso con la información disponible hasta el momento.

## Entrenamiento

Para la selección de hiperparámetros del modelo HAR, se aplica el mismo protocolo experimental detallado en las secciones anteriores. Se ejecuta una réplica bajo el esquema Walk-Forward, utilizando optimización bayesiana (Optuna) para explorar las longitudes de las ventanas temporales semanal ($L_w$) y mensual ($L_m$). Siguiendo la formulación original de Corsi (2009) y permitiendo cierta flexibilidad para adaptarse al mercado del Brent, $L_w$ explora los valores $\{5, 7\}$ días, mientras que $L_m$ explora $\{22, 30\}$ días. 

Se mantienen idénticos los criterios de control computacional (límite de 50 evaluaciones y parada temprana tras 15 intentos). En la práctica, dado que el HAR es un modelo lineal y el espacio de búsqueda se reduce a solo cuatro combinaciones posibles, la parada temprana se activa rápidamente al agotarse las combinaciones válidas.

A diferencia de los modelos GARCH y EGARCH, donde se aplica un factor de escalado a los retornos (multiplicación por 100) para garantizar la estabilidad numérica del optimizador de máxima verosimilitud (MLE), el HAR se estima mediante descomposición QR reducida (con respaldo en `lstsq`). Dado que la solución de mínimos cuadrados lineales es algebraicamente invariante a transformaciones de escala, no se aplicó ningún factor de escalado ni a los retornos ni a la varianza objetivo. Esto evita pasos innecesarios y mantiene la interpretabilidad directa de los coeficientes estimados. La naturaleza determinista de la resolución lineal asegura que no existan fallos de convergencia, a diferencia de lo observado en algunas iteraciones del GARCH.

In [2]:
def har_search_space(trial: optuna.Trial) -> dict:
    return {
        "lag_weekly": trial.suggest_categorical("lag_weekly", [5, 7]),
        "lag_monthly": trial.suggest_categorical("lag_monthly", [22, 30]),
        "seed": 42
    }

exec_id = 6
current_warmup = 1200 + (exec_id * 10)

exec_dir = RESULTS_PATH / f"exec_{exec_id}"
exec_dir.mkdir(parents=True, exist_ok=True)

trainer = WalkForwardTrainer(
    model_class=HARWrapper,
    scorer=VolatilityScorer(),
    output_dir=exec_dir,
    n_splits=5, 
    warmup=current_warmup,
    embargo=1, 
    n_trials=50,
    suggest_params_fn=har_search_space,
    early_stop_rounds=15
)

trainer.run(y_train, rs_train)

  0%|          | 0/50 [00:00<?, ?it/s]

La búsqueda identificó como configuración óptima al HAR con ventanas de 5 días semanales y 22 días mensuales ($L_w = 5$, $L_m = 22$), alcanzando un QLIKE de $0.6066$ ($0.606578$) en el ensayo 0 tras 16 evaluaciones completas de 50 posibles y 0 podados, con detención temprana al agotarse la exploración útil del espacio discreto. La ausencia de podas refleja que la resolución lineal mediante descomposición QR reducida (con respaldo en `lstsq`) opera de forma determinista desde la primera partición, sin fallos de convergencia ni inestabilidad numérica.

In [3]:
resumen_data = []

exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

tiempo_total_rep = 0.0
tiempo_promedio_fold = 0.0
num_folds = 0
total_trials, completos, podados, fallidos = 0, 0, 0, 0
best_params_str = "-"
mejor_qlike = "-"

csv_files = list(exec_dir.glob("*_results.csv"))
if csv_files:
    df_csv = pd.read_csv(csv_files[0])
    tiempo_total_rep = df_csv["train_time_sec"].sum()
    tiempo_promedio_fold = df_csv["train_time_sec"].mean()
    num_folds = len(df_csv)

db_path = exec_dir / "optuna_study.db"
if db_path.exists():
    try:
        storage_url = f"sqlite:///{db_path.resolve()}"
        study = optuna.load_study(study_name="HARWrapper_study", storage=storage_url)

        total_trials = len(study.trials)
        completos = len([t for t in study.trials if t.state == optuna.trial.TrialState.COMPLETE])
        podados  = len([t for t in study.trials if t.state == optuna.trial.TrialState.PRUNED])
        fallidos = len([t for t in study.trials if t.state == optuna.trial.TrialState.FAIL])

        mejor_qlike = round(study.best_value, 4)
        
        w_val = study.best_params.get('lag_weekly', '?')
        m_val = study.best_params.get('lag_monthly', '?')
        best_params_str = f"({w_val}, {m_val})"

    except Exception as e:
        print(f"Error leyendo BD en {exec_dir}: {e}")

resumen_data.append({
    "Total Trials": total_trials,
    "Completos": completos,
    "Podados": podados,
    "Mejor Config (Lw, Lm)": best_params_str,
    "Mejor QLIKE": mejor_qlike,
    "Tiempo Total (s)": round(tiempo_total_rep, 2),
    "Tiempo por Fold (s)": round(tiempo_promedio_fold, 3)
})

df_search_params = pd.DataFrame(resumen_data)
tt.display_styled_df(df=df_search_params, decimals=4, align_left_first=False)

Total Trials,Completos,Podados,"Mejor Config (Lw, Lm)",Mejor QLIKE,Tiempo Total (s),Tiempo por Fold (s)
16,16,0,"(5, 22)",0.6066,0.3600,0.0180


La búsqueda completa en Optuna tomó 29 segundos (a $3.14\text{ it/s}$), mientras que el ajuste y entrenamiento del modelo bajo el esquema *Walk-Forward* totalizó 0.3600 segundos, con un promedio de 0.0180 segundos por partición. Esta rapidez confirma la eficiencia computacional del modelo y su viabilidad para actualizaciones recurrentes sin requerir hardware especializado.

El desempeño de las configuraciones evaluadas muestra que la ventana mensual de 22 días supera con claridad a la de 30 días, lo que indica que la memoria relevante del mercado del Brent en el horizonte de largo plazo se concentra en torno al mes bursátil de 22 días hábiles, mientras que ampliar la ventana a 30 días incorpora información más antigua que degrada el pronóstico.

In [4]:
exec_id = 6
exec_dir = RESULTS_PATH / f"exec_{exec_id}"

df_har = pd.read_csv(exec_dir / "HARWrapper_results.csv")
cols_agrupar = ["lag_weekly", "lag_monthly"]
df_agg = df_har.groupby(cols_agrupar)["qlike"].agg(["mean", "std"]).reset_index()

df_agg["QLIKE"] = df_agg.apply(lambda x: f"{x['mean']:.3f} ± {x['std']:.3f}", axis=1)
df_top5 = df_agg.sort_values(by="mean").head(5)

df_top5 = df_top5.rename(columns={
    "lag_weekly": "Lag Semanal (Lw)",
    "lag_monthly": "Lag Mensual (Lm)"
})

df_final = df_top5[["Lag Semanal (Lw)", "Lag Mensual (Lm)", "QLIKE"]]
tt.display_styled_df(df=df_final, align_left_first=False, decimals=3)

Lag Semanal (Lw),Lag Mensual (Lm),QLIKE
5,22,0.607 ± 0.131
7,22,0.613 ± 0.129
7,30,0.790 ± 0.250
5,30,0.793 ± 0.260


La elección de los parámetros $(5, 22)$ responde tanto a la menor pérdida media de validación frente a $(7, 22)$ como a un criterio de parsimonia empírica. Dado que la diferencia entre ambas configuraciones es pequeña frente a la dispersión entre *folds*, mantener 5 días hábiles evita añadir rezagos adicionales sin ganancia clara en validación y preserva la correspondencia directa con la semana operativa de mercado y la formulación original de Corsi (2009).

In [5]:
best_model = HARWrapper(lag_weekly=5, lag_monthly=22)
best_model.fit(y_train, rs_train)

La estimación de los coeficientes del modelo HAR mostró que la componente mensual ejerció el mayor impacto sobre la varianza (0.3745), seguida de la diaria (0.3349) y, en menor medida, de la semanal (0.1396), lo que indica que la dinámica de largo plazo concentró el principal peso en la formación de la volatilidad del Brent. En términos de significancia estadística bajo errores estándar robustos HAC, las componentes mensual ($p = .009$) y semanal ($p = .035$) resultaron estadísticamente significativas, mientras que la diaria ($p = .072$) y la constante no resultó significativa ($p = .693$).

In [6]:
df_params_boot = best_model.get_parameter_table(
    B=2000, decimals=4, block_size=22, verbose=False
)

df_params_boot = df_params_boot.rename(columns={
    "Parameter": "Parámetro",
    "Coef": "Coeficiente",
    "Std. Error": "Error Estándar",
    "P-value": "p-valor",
    "CI Lower": "IC Inferior (95%)",
    "CI Upper": "IC Superior (95%)"
})

tt.display_styled_df(
    df=df_params_boot,
    decimals=4,
    p_value_cols=["p-valor"]
)

Parámetro,Coeficiente,Error Estándar,p-valor,IC Inferior (95%),IC Superior (95%)
const,0.0000,0.0001,.693,-0.0000,0.0001
RV_d,0.3349,0.1863,.072,0.3173,0.3540
RV_w,0.1396,0.0662,.035,0.0792,0.2059
RV_m,0.3745,0.1439,.009,0.2965,0.4772


La suma de los tres coeficientes de volatilidad (0.8490) resultó menor que uno, lo que refleja una alta persistencia de la volatilidad pero con reversión a la media en el largo plazo. Asimismo, los intervalos de confianza al 95 % obtenidos mediante *block bootstrap* no incluyeron el cero en ninguno de los tres rezagos, respaldando la estabilidad empírica de los estimadores.

A partir de las varianzas predichas y los retornos observados en el conjunto de entrenamiento, los residuos estandarizados en muestra se definen como:

$$
z_t = \frac{y_t}{\sqrt{\hat{\sigma}_t^2}}
$$

Para evaluar la estructura de estos residuos se examinó autocorrelación serial (Ljung-Box), heterocedasticidad condicional (ARCH-LM), normalidad (Jarque-Bera y Lilliefors) y dependencia no lineal (BDS). A diferencia de lo observado en las especificaciones GARCH y EGARCH, todos estos contrastes rechazaron la hipótesis nula en entrenamiento, lo que indica que la estructura lineal del HAR no absorbe por completo la autocorrelación residual, los efectos ARCH, la falta de normalidad ni la estructura no lineal de las innovaciones estandarizadas.

In [7]:
residuos_har = best_model.model_fit.resid
std_residuals = residuos_har / residuos_har.std()

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),133.8889,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 20),230.2469,< .001,< .001,Autocorrelación Significativa
Ljung-Box (Lag 30),394.3918,< .001,< .001,Autocorrelación Significativa
ARCH-LM (Lag 10),260.1875,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 20),277.8604,< .001,< .001,Efectos ARCH Significativos
ARCH-LM (Lag 30),393.3109,< .001,< .001,Efectos ARCH Significativos
Jarque-Bera,19485254.9514,< .001,--,Residuos No Normales
Lilliefors,0.3072,< .001,--,Residuos No Normales
BDS (Dim 2),14.9415,< .001,--,Estructura No Lineal Significativa


Este comportamiento puede vincularse tanto a la rigidez de los promedios móviles como al ruido del proxy empleado. Al asignar pesos fijos sobre las ventanas temporales, el HAR suaviza los cambios bruscos de régimen y carece de la actualización recursiva de varianza propia de los modelos GARCH. Asimismo, utilizar los retornos al cuadrado como proxy de varianza pasada introduce ruido de medición que puede manifestarse como dependencia no lineal en los residuos.

El análisis de la distribución de los residuos evidenció que la estructura del HAR no capturó adecuadamente la forma de los choques. La inspección visual del histograma reveló una distribución altamente leptocúrtica, caracterizada por una concentración extrema en el centro y colas muy pesadas. Esta falta de ajuste a la normalidad se confirmó estadísticamente (Lilliefors: $p < .001$), mientras que las pruebas de bondad de ajuste no rechazaron la distribución t-Student (Kolmogorov-Smirnov: $p = .399$; Cramér-von Mises: $p = .238$) Estos resultados evidenciaron que la distribución t-Student con grados de libertad muy bajos representó la forma empírica de los errores en entrenamiento, reflejando la fuerte concentración central y las colas pesadas de las innovaciones del mercado.

In [25]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 2),
    legend_loc="upper left"
)

La inspección visual del ajuste en la muestra de entrenamiento muestra que la serie estimada por el HAR sigue la trayectoria general y los ciclos de la volatilidad realizada sin desfases temporales evidentes. En el recuadro de la crisis de COVID-19 (2020-01 a 2020-07) y en episodios puntuales posteriores (como en 2022), el modelo reacciona ante los incrementos abruptos de volatilidad e incluso supera a la volatilidad realizada en picos específicos (como en marzo de 2020), aunque el máximo absoluto observado cerca de finales de abril de 2020 sigue siendo mayor en la serie realizada.

Fuera de los picos más agudos, persiste un suavizado característico de los promedios móviles y una asimetría en la magnitud de los shocks extremos, lo que se corresponde con las colas pesadas y la estructura leptocúrtica identificadas en los residuos estandarizados.

In [21]:
pred_varianza_har = best_model.model_fit.fittedvalues

lag = best_model.lag_monthly
rs_volatilidad_real = np.sqrt(rs_train[lag:])
pred_volatilidad_har = np.sqrt(pred_varianza_har)
fechas_alineadas = df_train.index[lag:]

mask_crisis = (fechas_alineadas >= "2020-01-01") & (fechas_alineadas <= "2020-06-30")

ts.plot_volatility_fit(
    dates=fechas_alineadas,
    rs_variance=rs_volatilidad_real,      
    pred_variance=pred_volatilidad_har,
    title="Ajuste dentro la Muestra de Entrenamiento (HAR 5,22)", 
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad HAR $\left(\hat{\sigma}_t\right)$",
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    zoom_slice=mask_crisis, 
    zoom_pad_frac=0.04,
    zoom_title="Zoom: Crisis COVID-19",
    zoom_bbox=(0.18, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Evaluación

La evaluación del HAR sigue el mismo protocolo experimental detallado en las secciones anteriores. El rendimiento se contrasta comparando el ajuste sobre los datos observados con la capacidad predictiva sobre datos no vistos. 

Para diagnosticar la presencia de sobreajuste (*overfitting*) o subajuste (*underfitting*), se mantiene el criterio de la diferencia absoluta entre las métricas de prueba y entrenamiento:

$$
\Delta_{abs} = M_{test} - M_{train}
$$

Al igual que en los modelos anteriores, un valor de $|\Delta_{abs}|$ que supere el umbral de tolerancia de 0.05 evidencia falta de generalización. 

Las predicciones fuera de muestra se generan ejecutando el método `forecast`, el cual aplica un esquema de ventana expansiva (*rolling forward*) que reestima los coeficientes OLS en cada paso para garantizar la ausencia de fuga de información. A diferencia de los modelos GARCH y EGARCH, el HAR no requiere factores de escalado numérico para su estabilidad, por lo que las varianzas pronosticadas se comparan directamente con la volatilidad realizada en su escala original.

In [11]:
start_time = time.time()

preds_test = best_model.forecast(y_seq=y_test, rs_seq=rs_test)
end_time = time.time()
tiempo_evaluacion = end_time - start_time


df_preds = pd.DataFrame({
    "date": df_test.index, 
    "rs_true": rs_test, 
    "y_pred": preds_test
})

output_path = RESULTS_PATH / "best_har_predictions.csv"
df_preds.to_csv(output_path, index=False)

La generación de pronósticos recursivos de un paso adelante fuera de muestra sobre las 953 observaciones del conjunto de prueba tomó 6.54 segundos (aproximadamente 0.0069 segundos por predicción). Este rendimiento confirma la eficiencia computacional de la actualización iterativa del modelo lineal frente al reentrenamiento de arquitecturas de *Deep Learning*.

El diagnóstico de los residuos estandarizados en el cojunto de prueba muestra que las pruebas de Ljung-Box no rechazan la hipótesis nula de ausencia de autocorrelación serial en ninguno de los rezagos evaluados, lo que indica que el modelo HAR absorbió la dependencia temporal lineal en datos no vistos. Los contrastes ARCH-LM no resultan significativos tras el ajuste por multiplicidad, por lo que no se evidencia agrupamiento de varianza condicional residual en el periodo de prueba.

In [12]:
std_residuals_oos = y_test / np.sqrt(preds_test)
std_residuals_oos = std_residuals_oos[np.isfinite(std_residuals_oos)]

df_diag = tt.run_model_diagnostics(
    std_residuals=std_residuals_oos,
    lags=(10, 20, 30),
    p_adjust="bonferroni",
    alpha=0.05,
    english=False,
    column_titles=(
        "Prueba Estadística", "Estadístico",
        "p-valor", "p-adj", "Conclusión (α = 0.05)")
)

df_display = df_diag.copy()

df_display["Estadístico"] = df_display["Estadístico"].apply(
    lambda x: f"{x:.4f}" if pd.notna(x) else "--")

def _format_p(x):
    if pd.isna(x): return "--"
    if x < 0.001: return "< .001"
    formatted = f"{x:.3f}"
    return formatted[1:] if formatted.startswith("0.") else formatted

df_display["p-valor"] = df_display["p-valor"].apply(_format_p)
df_display["p-adj"] = df_display["p-adj"].apply(_format_p)

tt.display_styled_df(df=df_display, p_value_cols=[])

Prueba Estadística,Estadístico,p-valor,p-adj,Conclusión (α = 0.05)
Ljung-Box (Lag 10),8.8893,.543,1.000,Sin Autocorrelación
Ljung-Box (Lag 20),23.1039,.284,.851,Sin Autocorrelación
Ljung-Box (Lag 30),26.5898,.645,1.000,Sin Autocorrelación
ARCH-LM (Lag 10),15.1389,.127,.381,Sin Efectos ARCH
ARCH-LM (Lag 20),21.6190,.362,1.000,Sin Efectos ARCH
ARCH-LM (Lag 30),24.9836,.726,1.000,Sin Efectos ARCH
Jarque-Bera,200.9556,< .001,--,Residuos No Normales
Lilliefors,0.0395,.002,--,Residuos No Normales
BDS (Dim 2),-5.4970,< .001,--,Estructura No Lineal Significativa


Por el contrario, las pruebas de Jarque-Bera y Lilliefors rechazan la hipótesis de normalidad de los residuos estandarizados, y el contraste BDS (dimensión 2) vuelve a indicar dependencia no lineal significativa en las innovaciones. Aunque el conjunto de prueba presenta menor volatilidad extrema que el periodo de entrenamiento, persisten desviaciones de la normalidad y patrones no lineales que la especificación lineal aditiva del HAR no captura por completo.

El análisis de la distribución de los residuos fuera de muestra reveló un ajuste adecuado a la distribución t-Student (Kolmogorov-Smirnov: $p = .835$; Cramér-von Mises: $p = .888$). La prueba de razón de verosimilitud (estadístico $53.15$, $p < .001$) confirmó que la especificación t-Student supera de forma significativa a la distribución normal, lo cual se ve respaldado por una mayor log-verosimilitud y menores criterios de información AIC y BIC.

In [20]:
_= tst.distribution_fit_table(
    pd.Series(std_residuals_oos),
    column_titles=(
        "Distribución", 
        "Log-Likelihood", 
        "AIC", 
        "BIC", 
        "Estadístico LRT", 
        "p-valor LRT"
    )
)

Distribución,Log-Likelihood,AIC,BIC,Estadístico LRT,p-valor LRT
Normal,"-1,503.35","3,010.70","3,020.42",-,-
t-Student,"-1,476.77","2,959.54","2,974.12",53.15,< .001


La inspección visual corroboró estos resultados estadísticos. El histograma de los residuos mostró un ajuste preciso a la curva teórica de la t-Student ($\nu = 6.0896$), y el gráfico Q-Q evidenció que los cuantiles muestrales se alinearon con la línea diagonal de referencia en la mayor parte de la distribución, con ligeras desviaciones en los extremos. En conjunto, esta evidencia indicó que el HAR capturó la forma de los choques de manera adecuada en el conjunto de prueba y que su comportamiento respondió al régimen de volatilidad del periodo evaluado.

In [13]:
_, _ = tst.plot_distribution_fit(
    pd.Series(std_residuals_oos),
    dist="t",
    titles=("Distribución de Residuos", "Gráfico Q-Q (normal)"),
    xlab="Residuo estandarizado",
    ylab_left="Densidad",
    xlab_right="Cuantiles teóricos",
    ylab_right="Cuantiles muestrales",
    ylim_left=(0, 0.6),
    legend_loc="upper left"
)

La evaluación fuera de muestra mostró un comportamiento estable en la métrica principal QLIKE, con un incremento moderado entre entrenamiento y prueba que se mantuvo dentro del margen de estabilidad operativa. Las métricas puntuales (MSE, RMSE y MAE) y probabilísticas (CRPS y Pinball en los cuantiles 0.05 y 0.95) presentaron cambios mínimos entre ambos periodos, lo que indica consistencia en el ajuste promedio y en la calibración cuantílica.

Por su parte, las métricas ajustadas por heterocedasticidad (HMSE y HMAE) registraron aumentos moderados al pasar de entrenamiento a prueba, en línea con la sensibilidad de estas medidas frente a variaciones relativas de volatilidad en observaciones puntuales de mayor tensión.

In [14]:
preds_train_var = best_model.model_fit.fittedvalues
preds_test_var = df_preds["y_pred"].values

lag = best_model.lag_monthly
rs_train_aligned = rs_train[lag:]

ts.display_train_test_comparison(
    rs_train=rs_train_aligned,
    preds_train=preds_train_var,
    rs_test=rs_test,
    preds_test=preds_test_var,
    column_titles=(
        "Métrica", 
        "Entrenamiento", 
        "Prueba", 
        "Diferencia"
    )
)

Métrica,Entrenamiento,Prueba,Diferencia
QLIKE,0.4273,0.4471,+0.0198
CRPS,0.0128,0.0124,-0.0004
Pinball (τ=0.05),0.0027,0.0026,-0.0001
Pinball (τ=0.95),0.0014,0.0016,+0.0002
MSE,0.0000,0.0000,+0.0000
RMSE,0.0017,0.0022,+0.0005
MAE,0.0004,0.0004,+0.0000
HMSE,2.4412,3.3125,+0.8713
HMAE,0.7775,0.7970,+0.0196


La inspección visual de los pronósticos fuera de muestra muestra que la volatilidad estimada por el HAR (5, 22) sigue la trayectoria general de la volatilidad realizada durante los periodos de calma y moderación, mientras que en el recuadro del pico de volatilidad de 2026 (2026-02 a 2026-07) la curva predicha no alcanza el máximo registrado en marzo de 2026 y se mantiene por debajo de la serie realizada; como hipótesis, esta subestimación en los picos agudos podría asociarse al suavizado de las ventanas de promedios móviles y a la respuesta retardada de la estructura lineal ante cambios bruscos de régimen.

In [22]:
rs_test_vol = np.sqrt(df_preds["rs_true"])
pred_test_vol = np.sqrt(df_preds["y_pred"])

mask_crisis = (df_preds["date"] >= "2026-02-01") & (df_preds["date"] <= "2026-06-30")

ts.plot_volatility_fit(
    dates=df_preds["date"],
    rs_variance=rs_test_vol,      
    pred_variance=pred_test_vol,
    title="Ajuste fuera de la Muestra de Entrenamiento (HAR 5, 22)",
    ylab="Volatilidad",              
    label_realized=r"Volatilidad Realizada $\left(\sigma_R\right)$", 
    label_pred=r"Volatilidad HAR $\left(\hat{\sigma}_t\right)$", 
    
    ylim=(-0.01, 0.35), 
    legend_bbox=(0.01, 0.98), 
    
    zoom_slice=mask_crisis, zoom_pad_frac=0.04,
    zoom_title="Zoom: Pico de Volatilidad (2026)",
    zoom_bbox=(0.25, 0.35, 0.40, 0.40),
    title_size=14, label_size=14, tick_size=12, legend_size=14
)

## Conclusiones

La evaluación del modelo HAR con ventanas de 5 días semanales y 22 días mensuales ($L_w = 5$, $L_m = 22$) mostró que la estructura lineal aditiva captura la trayectoria general de la volatilidad del Brent con alta eficiencia computacional mediante descomposición QR reducida y errores robustos HAC. En validación y en el conjunto de prueba, la ventana mensual de 22 días superó a la de 30 días, en línea con la escala mensual de la Hipótesis de Mercado Heterogéneo de Corsi (2009), mientras que la estimación de coeficientes evidenció mayor peso relativo de las componentes mensual y semanal frente a la diaria, con persistencia elevada y reversión a la media.

En el conjunto de prueba, el modelo mantuvo un desempeño estable en QLIKE ($0.4471$ frente a $0.4273$ en entrenamiento), en las métricas puntuales y probabilísticas, y en las medidas ajustadas por heterocedasticidad (HMSE y HMAE). Sin embargo, la inspección visual del pico de 2026 muestra que la serie predicha no alcanza la magnitud máxima observada en marzo de 2026, lo que como hipótesis puede vincularse al suavizado propio de los promedios móviles fijos y a la respuesta retardada ante saltos bruscos de varianza.

En el análisis de residuos estandarizados fuera de muestra, los contrastes de Ljung-Box y ARCH-LM no mostraron autocorrelación serial ni efectos ARCH significativos tras el ajuste por comparaciones múltiples, y el ajuste a una distribución *t*-Student ($\nu = 6.0896$, ubicación $0.0353$, escala $0.9613$) resultó respaldado por las pruebas de Kolmogorov-Smirnov, Cramér-von Mises y la razón de verosimilitud frente a la normal. No obstante, persisten desviaciones de normalidad y dependencia no lineal en el contraste BDS, lo que indica que el modelo lineal ofrece una base parsimoniosa y rápida, pero deja espacio para arquitecturas capaces de capturar dinámicas no lineales y cambios abruptos de régimen.